In [1]:
import pandas as pd
import numpy as np
import os

train_raw = pd.read_csv('../data/train.csv')
test_raw = pd.read_csv('../data/test.csv')

passenger_ids_test = test_raw['PassengerId']

In [2]:
for df in [train_raw, test_raw]:
    df['Deck'] = df['Cabin'].str[0]
    df['Deck'] = df['Deck'].fillna('Unknown')
    df['Deck'] = df['Deck'].replace('T', 'Unknown')
    df['HasCabin'] = df['Cabin'].notnull().astype(int)
    df.drop(columns=['Cabin'], inplace=True)

In [3]:
mediana_age_train = train_raw.groupby(['Pclass', 'Sex'])['Age'].median()

def imputar_age(row, medianas):
    if pd.isnull(row['Age']):
        return medianas.loc[row['Pclass'], row['Sex']]
    return row['Age']

train_raw['Age'] = train_raw.apply(lambda row: imputar_age(row, mediana_age_train), axis=1)
test_raw['Age'] = test_raw.apply(lambda row: imputar_age(row, mediana_age_train), axis=1)

print(f"Nulos en train: {train_raw['Age'].isnull().sum()}, en test: {test_raw['Age'].isnull().sum()}")

Nulos en train: 0, en test: 0


In [4]:
mediana_age_train = train_raw.groupby(['Pclass', 'Sex'])['Age'].median()

def imputar_age(row, medianas):
    if pd.isnull(row['Age']):
        return medianas.loc[row['Pclass'], row['Sex']]
    return row['Age']

train_raw['Age'] = train_raw.apply(lambda row: imputar_age(row, mediana_age_train), axis=1)
test_raw['Age'] = test_raw.apply(lambda row: imputar_age(row, mediana_age_train), axis=1)

print(f"Nulos en train: {train_raw['Age'].isnull().sum()}, en test: {test_raw['Age'].isnull().sum()}")

Nulos en train: 0, en test: 0


In [5]:
moda_embarked_train = train_raw['Embarked'].mode()[0]
mediana_fare_train = train_raw['Fare'].median()

train_raw['Embarked'] = train_raw['Embarked'].fillna(moda_embarked_train)
test_raw['Embarked'] = test_raw['Embarked'].fillna(moda_embarked_train)

train_raw['Fare'] = train_raw['Fare'].fillna(mediana_fare_train)
test_raw['Fare'] = test_raw['Fare'].fillna(mediana_fare_train)

In [6]:
for df in [train_raw, test_raw]:
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

In [7]:
title_map = {
    'Mr': 'Mr', 'Miss': 'Miss', 'Mrs': 'Mrs', 'Master': 'Master',
    'Dr': 'Rare', 'Rev': 'Rare', 'Col': 'Rare', 'Major': 'Rare',
    'Mlle': 'Miss', 'Countess': 'Rare', 'Ms': 'Miss', 'Lady': 'Rare',
    'Jonkheer': 'Rare', 'Don': 'Rare', 'Dona': 'Rare', 'Capt': 'Rare', 'Sir': 'Rare'
}

for df in [train_raw, test_raw]:
    df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')
    df['Title'] = df['Title'].map(title_map)
    df['Title'] = df['Title'].fillna('Rare')

In [8]:
# AgeBin usa bins fijos (no depende de datos), es seguro directo
for df in [train_raw, test_raw]:
    df['AgeBin'] = pd.cut(df['Age'], bins=[0, 12, 18, 35, 60, 100], labels=['Niño', 'Adolescente', 'Adulto', 'Adulto_mayor', 'Anciano'])

# FareBin usa qcut (cuantiles), hay que definir los cortes SOLO con train
_, bins_fare = pd.qcut(train_raw['Fare'], 4, labels=['Baja', 'Media', 'Alta', 'MuyAlta'], retbins=True)

train_raw['FareBin'] = pd.qcut(train_raw['Fare'], 4, labels=['Baja', 'Media', 'Alta', 'MuyAlta'])
test_raw['FareBin'] = pd.cut(test_raw['Fare'], bins=bins_fare, labels=['Baja', 'Media', 'Alta', 'MuyAlta'], include_lowest=True)

In [9]:
def ticket_item(x):
    items = x.split(" ")
    if len(items) == 1:
        return "NONE"
    return "_".join(items[0:-1]).replace(".", "").replace("/", "").upper()

train_raw['Ticket_item'] = train_raw['Ticket'].apply(ticket_item)
test_raw['Ticket_item'] = test_raw['Ticket'].apply(ticket_item)

conteo_tickets_train = train_raw['Ticket_item'].value_counts()
categorias_frecuentes = conteo_tickets_train[conteo_tickets_train >= 10].index

train_raw['Ticket_item'] = train_raw['Ticket_item'].apply(lambda x: x if x in categorias_frecuentes else 'RARE')
test_raw['Ticket_item'] = test_raw['Ticket_item'].apply(lambda x: x if x in categorias_frecuentes else 'RARE')

In [10]:
for df in [train_raw, test_raw]:
    df.drop(columns=['Name', 'Ticket', 'Age'], inplace=True)

train_raw.drop(columns=['PassengerId'], inplace=True)
test_raw.drop(columns=['PassengerId'], inplace=True)  # ya guardamos passenger_ids_test aparte

train_raw.isnull().sum()

Survived       0
Pclass         0
Sex            0
SibSp          0
Parch          0
Fare           0
Embarked       0
Deck           0
HasCabin       0
FamilySize     0
IsAlone        0
Title          0
AgeBin         0
FareBin        0
Ticket_item    0
dtype: int64

In [11]:
# Sex: male/female -> 0/1
train_raw['Sex'] = train_raw['Sex'].map({'male': 0, 'female': 1})
test_raw['Sex'] = test_raw['Sex'].map({'male': 0, 'female': 1})

# One-hot encoding por separado
train_encoded = pd.get_dummies(train_raw, columns=['Embarked', 'Title', 'AgeBin', 'FareBin', 'Ticket_item', 'Deck'], drop_first=True)
test_encoded = pd.get_dummies(test_raw, columns=['Embarked', 'Title', 'AgeBin', 'FareBin', 'Ticket_item', 'Deck'], drop_first=True)

# Guardar PassengerId dentro de test_encoded ANTES de alinear columnas
test_encoded['PassengerId'] = passenger_ids_test.values

# Alinear columnas: test debe tener EXACTAMENTE las mismas columnas que train (sin Survived)
# pero conservando PassengerId aparte
train_cols = train_encoded.drop(columns=['Survived']).columns
test_encoded_features = test_encoded.reindex(columns=train_cols, fill_value=0)
test_encoded_features['PassengerId'] = test_encoded['PassengerId'].values

print(f"Columnas en train (sin Survived): {len(train_cols)}")
print(f"Columnas en test (con PassengerId): {len(test_encoded_features.columns)}")
print(f"¿Coinciden features?: {list(train_cols) == [c for c in test_encoded_features.columns if c != 'PassengerId']}")

Columnas en train (sin Survived): 36
Columnas en test (con PassengerId): 37
¿Coinciden features?: True


In [12]:
os.makedirs('../data/processed', exist_ok=True)
train_encoded.to_csv('../data/processed/train_clean.csv', index=False)
test_encoded.to_csv('../data/processed/test_clean.csv', index=False)
print("Archivos guardados correctamente ✅")

Archivos guardados correctamente ✅
